In [ ]:
from pyspark.sql import SparkSession

# Создание SparkSession
spark = SparkSession.builder.appName("web_server_logs").getOrCreate()

logs_df = spark.read.csv("web_server_logs.csv", header=True)
logs_df.createOrReplaceTempView("web_server_logs")


# Сгруппируйте данные по IP и посчитайте количество запросов для каждого IP, 
# выводим 10 самых активных IP.
task_1 = spark.sql("""
SELECT ip, request_count
FROM (SELECT ip, count(*) AS request_count
    FROM web_server_logs
    GROUP BY ip)
ORDER BY request_count DESC
LIMIT 10
""")

# Сгруппируйте данные по HTTP-методу и посчитайте количество запросов для каждого метода
task_2 = spark.sql("""
SELECT method, method_count
FROM (SELECT method, count(*) AS method_count
    FROM web_server_logs
    GROUP BY method)
""")

# Профильтруйте и посчитайте количество запросов с кодом ответа 404
task_3 = spark.sql("""
SELECT count(*) FROM web_server_logs WHERE response_code = 404
""")

# Сгруппируйте данные по дате и просуммируйте размер ответов, сортируйте по дате
task_4 = spark.sql("""
SELECT date, INT(total_response_size) as total_response_size FROM (
SELECT DATE(timestamp) as date, SUM(response_size) AS total_response_size
FROM web_server_logs
GROUP BY DATE(timestamp))
ORDER BY date
""")

task_1.show()

task_2.show()

print(f"Number of 404 response code: {task_3.collect()[0][0]}\n")

print(f"Total response size by day:")
task_4.show()

+---------------+-------------+
|             ip|request_count|
+---------------+-------------+
|  12.69.106.112|            2|
|  13.111.222.76|            2|
| 70.171.151.133|            1|
|153.119.232.121|            1|
| 182.219.163.31|            1|
|216.188.223.102|            1|
|   61.221.58.63|            1|
| 18.174.227.161|            1|
|  198.247.73.63|            1|
|214.102.216.195|            1|
+---------------+-------------+

+------+------------+
|method|method_count|
+------+------------+
|  POST|       25112|
|DELETE|       24760|
|   PUT|       24812|
|   GET|       25316|
+------+------------+

Number of 404 response code: 24835

Total response size by day:
+----------+-------------------+
|      date|total_response_size|
+----------+-------------------+
|2026-01-01|            9059270|
|2026-01-02|            9548149|
|2026-01-03|            8818338|
|2026-01-04|            9221776|
|2026-01-05|            9699173|
|2026-01-06|            9199791|
|2026-01-07| 